# 07 - Tokenizer (AI Infra 视角)

本节从 **工程实现** 角度理解分词器：
- 分词器性能对比
- 词表大小与效率权衡
- 特殊 Token 设计
- Rust 加速实现

In [ ]:
import torch
import time

## 1. Tokenizer 核心 (30秒版)

```
文本 "Hello World"
        ↓ encode
  [15496, 2159]
        ↓ 模型处理
   [42, 123, ...]
        ↓ decode  
   "The answer"

关键指标:
  - 编码速度: tokens/sec
  - 压缩率: chars/token
  - 词表大小: 影响 Embedding 参数量
```

## 2. 分词器性能对比

| 分词器 | 语言 | 速度 | 使用场景 |
|--------|------|------|----------|
| HuggingFace | Python | 1x | 开发测试 |
| tiktoken | Python+Rust | 3-5x | OpenAI API |
| **RustBPE** | **Rust** | **10x+** | **nanochat** |
| SentencePiece | C++ | 5x | LLaMA |

In [ ]:
# 性能测试
try:
    from transformers import AutoTokenizer
    import tiktoken
    
    # 测试文本
    text = "Hello, World! " * 1000
    
    # HuggingFace
    hf_tok = AutoTokenizer.from_pretrained("gpt2")
    start = time.time()
    for _ in range(100):
        hf_tok.encode(text)
    hf_time = time.time() - start
    
    # tiktoken
    tt_tok = tiktoken.get_encoding("gpt2")
    start = time.time()
    for _ in range(100):
        tt_tok.encode(text)
    tt_time = time.time() - start
    
    print(f"编码 {len(text)} 字符 × 100 次:")
    print(f"  HuggingFace: {hf_time:.2f}s")
    print(f"  tiktoken:    {tt_time:.2f}s ({hf_time/tt_time:.1f}x faster)")
    
except ImportError as e:
    print(f"需要安装: {e}")

## 3. 词表大小权衡 (重要!)

```
小词表 (32K):              大词表 (128K):
  + Embedding 参数少         + 序列更短
  + 训练数据覆盖好           + 推理更快
  - 序列更长                 - Embedding 参数多
  - 推理慢                   - 需要更多训练数据

Embedding 参数: vocab × dim
  32K × 4096 = 131M
  128K × 4096 = 524M (多 4x!)
```

In [ ]:
# 词表大小对比
vocab_configs = {
    "GPT-2":      50257,
    "LLaMA 1/2":  32000,
    "LLaMA 3":    128256,
    "GPT-4":      100277,
    "Qwen":       151936,
}

dim = 4096
print(f"Embedding 参数量 (dim={dim}):")
print("-" * 40)
for name, vocab in vocab_configs.items():
    params = vocab * dim
    print(f"{name:12s}: vocab={vocab:6d}, params={params/1e6:.0f}M")

In [ ]:
# 中文效率对比
try:
    import tiktoken
    
    gpt2 = tiktoken.get_encoding("gpt2")
    cl100k = tiktoken.get_encoding("cl100k_base")
    
    chinese_text = "人工智能是计算机科学的一个重要分支，它试图理解智能的本质。"
    
    gpt2_tokens = gpt2.encode(chinese_text)
    cl100k_tokens = cl100k.encode(chinese_text)
    
    print(f"中文: {chinese_text}")
    print(f"字符数: {len(chinese_text)}")
    print(f"\nGPT-2 (50K vocab):   {len(gpt2_tokens)} tokens")
    print(f"cl100k (100K vocab): {len(cl100k_tokens)} tokens")
    print(f"\n效率提升: {len(gpt2_tokens)/len(cl100k_tokens):.1f}x")
    
except ImportError:
    print("需要安装 tiktoken")

## 4. BPE 算法要点

```
Byte-Level BPE:
  1. 从 256 个字节开始
  2. 统计相邻 pair 频率
  3. 合并最频繁的 pair
  4. 重复直到达到目标词表大小

优点:
  - 可处理任何语言/字符
  - 无 OOV (Out-of-Vocabulary)
  - 常见词变成单个 token
```

In [ ]:
# BPE 合并演示
def simple_bpe_demo(text, n_merges=3):
    tokens = list(text)
    print(f"初始: {tokens}")
    
    for i in range(n_merges):
        # 统计 pair 频率
        pairs = {}
        for j in range(len(tokens) - 1):
            pair = (tokens[j], tokens[j+1])
            pairs[pair] = pairs.get(pair, 0) + 1
        
        if not pairs:
            break
        
        # 找最频繁的
        best_pair = max(pairs, key=pairs.get)
        new_token = best_pair[0] + best_pair[1]
        
        # 合并
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and (tokens[j], tokens[j+1]) == best_pair:
                new_tokens.append(new_token)
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        tokens = new_tokens
        
        print(f"合并 {best_pair} → '{new_token}': {tokens}")

simple_bpe_demo("aaabdaaabac", n_merges=4)

## 5. 特殊 Token 设计

```
nanochat 的特殊 Token:

<|bos|>              序列开始
<|user_start|>       用户消息开始
<|user_end|>         用户消息结束
<|assistant_start|>  助手消息开始
<|assistant_end|>    助手消息结束
<|python_start|>     Python 代码开始
<|python_end|>       Python 代码结束
<|output_start|>     输出开始
<|output_end|>       输出结束
```

In [ ]:
# 对话渲染示例
def render_conversation(messages):
    """把对话渲染成带特殊 token 的格式"""
    result = "<|bos|>"
    for msg in messages:
        if msg["role"] == "user":
            result += f"<|user_start|>{msg['content']}<|user_end|>"
        else:
            result += f"<|assistant_start|>{msg['content']}<|assistant_end|>"
    return result

messages = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！有什么可以帮你的？"},
]

print("渲染结果:")
print(render_conversation(messages))

## 6. 训练时的 Mask

```
SFT 训练只计算 assistant 回复的 loss:

<|bos|><|user|>你好<|/user|><|assistant|>你好！<|/assistant|>
  0       0     0      0          0         1      1
                                            ↑
                                    只有这部分计算 loss
```

In [ ]:
def create_loss_mask(messages):
    """创建 loss mask: 只对 assistant 消息计算 loss"""
    tokens = ["<|bos|>"]
    mask = [0]
    
    for msg in messages:
        if msg["role"] == "user":
            tokens.extend(["<|user_start|>", msg["content"], "<|user_end|>"])
            mask.extend([0, 0, 0])
        else:
            tokens.extend(["<|assistant_start|>", msg["content"], "<|assistant_end|>"])
            mask.extend([0, 1, 1])  # 内容和结束符计算 loss
    
    return tokens, mask

tokens, mask = create_loss_mask(messages)
print("Token 和 Mask:")
for t, m in zip(tokens, mask):
    status = "计算loss" if m else "忽略"
    print(f"  [{status:6s}] {t}")

## 7. nanochat RustBPE

nanochat 使用 Rust 实现的高性能分词器:

```rust
// rustbpe/src/lib.rs
pub struct Tokenizer {
    encoder: HashMap<Vec<u8>, u32>,
    decoder: HashMap<u32, Vec<u8>>,
    bpe_ranks: HashMap<(Vec<u8>, Vec<u8>), u32>,
}
```

优势:
- 比 Python 快 10x+
- 内存效率高
- PyO3 绑定，Python 无缝调用

In [ ]:
# 尝试加载 RustBPE
try:
    import rustbpe
    print("RustBPE 已安装!")
    print(f"版本: {rustbpe.__version__ if hasattr(rustbpe, '__version__') else 'N/A'}")
except ImportError:
    print("RustBPE 未安装")
    print("安装: uv run maturin develop -r")

## 8. 面试常见问题

### Q1: BPE 和 WordPiece 的区别?

**答**:
- BPE: 基于频率合并 pair
- WordPiece: 基于似然度选择合并
- 实际效果差异不大，BPE 更常用

---

### Q2: 为什么用 Byte-Level BPE?

**答**:
- 从 256 字节开始，可处理任何 UTF-8 文本
- 无 OOV 问题
- 不需要预定义字符集

---

### Q3: 词表大小如何选择?

**答**:
- 32K: 参数少，适合小模型
- 100K+: 压缩率高，适合多语言
- 权衡: Embedding 参数 vs 序列长度

---

### Q4: 分词器如何影响模型性能?

**答**:
- 压缩率高 → 序列短 → 处理更多上下文
- 中文友好 → 不会把汉字拆成多个字节
- 特殊 token → 支持工具调用、多角色对话

---

### Q5: 为什么用 Rust 实现分词器?

**答**:
- CPU 密集型任务，Python 太慢
- 数据预处理是训练瓶颈之一
- PyO3 可以无缝集成到 Python

---

### Q6: Pre-tokenization 是什么?

**答**:
- BPE 之前的预切分
- 防止跨词边界合并 (如 "dog." 和 "dog,")
- 用正则表达式切分

## 9. 总结速查表

| 主题 | 要点 |
|------|------|
| **BPE** | 字节级，频率合并，无 OOV |
| **词表大小** | 32K-150K，影响 Embedding 参数 |
| **特殊 Token** | 标记角色、代码、输出边界 |
| **Loss Mask** | 只对 assistant 回复计算 loss |
| **性能** | Rust > tiktoken > HuggingFace |

### 速度经验值

```
Python (HuggingFace): ~100K tokens/sec
tiktoken:             ~500K tokens/sec  
Rust (RustBPE):       ~1M+ tokens/sec
```